# YOLOv11 Model Training
- Kaggle was used for training with T4 x2 GPUs

# CC0 1.0 Universal (Public Domain Dedication)
# Copyright (C) 2025 JustSplash8501
# This work is dedicated to the public domain under the CC0 1.0 Universal license.
# You can copy, modify, distribute, and perform the work, even for commercial purposes,
# without asking permission. No rights are reserved.
# Full license text: https://creativecommons.org/publicdomain/zero/1.0/legalcode.txt

In [ ]:
import boto3
from pathlib import Path
import zipfile
import sagemaker
from sagemaker.pytorch import PyTorch

# Initialize SageMaker session
sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name

# Define local paths
zip_path = Path("/opt/ml/input/data/Pistols.v1-resize-416x416.yolov11.zip")
extract_folder = Path("/opt/ml/input/data/extracted/")
extract_folder.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("Dataset extracted to:", extract_folder)


# Prepare Dataset

In [ ]:
import random
import shutil
import yaml

def prepare_yolo_dataset(images_dir, labels_dir, output_dir,
                         train_ratio=0.85, val_ratio=0.15,
                         seed=42, class_names=['pistol']):
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)
    output_dir = Path(output_dir)
    random.seed(seed)

    # Create folder structure
    splits_dirs = {}
    for split in ["train", "val"]:
        splits_dirs[split] = {
            "images": output_dir / split / "images",
            "labels": output_dir / split / "labels"
        }
        for d in splits_dirs[split].values():
            d.mkdir(parents=True, exist_ok=True)

    # Get all label files
    label_files = list(labels_dir.glob("*.txt"))
    random.shuffle(label_files)
    n_total = len(label_files)
    n_train = int(train_ratio * n_total)

    splits = {
        "train": label_files[:n_train],
        "val": label_files[n_train:]
    }

    # Copy labels and images
    for split_name, files in splits.items():
        for lbl in files:
            shutil.copy(lbl, splits_dirs[split_name]["labels"] / lbl.name)
            img_file = images_dir / f"{lbl.stem}.jpg"
            if not img_file.exists():
                raise FileNotFoundError(f"Image not found: {img_file}")
            shutil.copy(img_file, splits_dirs[split_name]["images"] / img_file.name)

    # Create data.yaml
    data_yaml = {
        "train": str(output_dir / "train" / "images"),
        "val": str(output_dir / "val" / "images"),
        "nc": len(class_names),
        "names": class_names
    }

    yaml_path = output_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
    
    print(f"Dataset prepared at {output_dir}")
    print(f"Train: {len(splits['train'])}, Val: {len(splits['val'])}")
    return yaml_path

# Prepare dataset
data_yaml_path = prepare_yolo_dataset(
    images_dir=extract_folder / "export/images",
    labels_dir=extract_folder / "export/labels",
    output_dir="/opt/ml/input/data/dataset"
)


# Upload Dataset

In [ ]:
s3_dataset_prefix = "yolov11/pistols/dataset"
s3_dataset_path = sagemaker_session.upload_data(
    path="/opt/ml/input/data/dataset",
    bucket=bucket,
    key_prefix=s3_dataset_prefix
)
print("Dataset uploaded to:", s3_dataset_path)


# Run Model

In [ ]:
import argparse
from pathlib import Path
from ultralytics import YOLO
import torch

parser = argparse.ArgumentParser()
parser.add_argument('--data', type=str, required=True)
parser.add_argument('--epochs', type=int, default=100)
parser.add_argument('--imgsz', type=int, default=640)
parser.add_argument('--batch', type=int, default=32)
parser.add_argument('--device', type=str, default='cuda')
args = parser.parse_args()

device = args.device if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

# Load pretrained model
model = YOLO("yolo11n.pt")

# Train model
model.train(
    data=args.data,
    epochs=args.epochs,
    imgsz=args.imgsz,
    batch=args.batch,
    device=device,
    name="pistol_model_v1_0",
    pretrained=True,
    amp=True,
    half=True,
    val=True,
    workers=4,
    save_period=10,
    mosaic=True,
    mixup=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.1,
    shear=2.0,
    fliplr=0.5,
    patience=10,
    cos_lr=True
)
